<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_02_graph_basics_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 02 — Graphs from Scratch

**Deep Learning for Engineering · Aalborg University · Part 1**

Notebook 01 applied a learned local rule on a grid. A grid is a very particular
graph: every pixel has the same four neighbours, in the same directions, at the
same distance. Take those regularities away and the convolution stops making
sense — but the *idea* survives, and what survives is a graph neural network.

This notebook builds every piece of that idea in NumPy before any library is
allowed near it. By the end you will have written message passing yourself, in
about five lines, and checked that relabelling the buses cannot change its answer.

There is no training here and no `torch`. That is deliberate. A graph network is
usually met as a library call, and students then carry a vague picture of
"something with neighbours" into a problem where the details decide whether it
works. The details are all in this notebook.

The graph is the **six-bus power network** you will train on in notebook 03.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import Ex_5_core as core

np.set_printoptions(precision=3, suppress=True)

edges = core.six_bus_edges()
print("buses:", len(core.BUS_NAMES))
for i, name in enumerate(core.BUS_NAMES):
    print(f"  {i}  {name}")
print("\nlines, as impedance z = R + jX and admittance y = 1/z (per unit, 100 MVA base):\n")
print(core.line_table())

core.plot_graph(title="The six-bus network")
plt.show()

**What you should see.** Six buses numbered 0 to 5, a table of the eight lines
with their resistance R, reactance X and admittance y, and the network drawn with
bus 0 on the left and bus 5 on the right.

Take a moment over the picture, because notebook 03 assumes you know it. Buses 0
and 1 are generation. Buses 2, 3 and 4 are load centres. Bus 5 is an HVDC link
arriving from a neighbouring system, which appears to this network as a fixed
injection — it is not a machine and it does not respond to frequency. The
topology is loosely in the spirit of a Danish transmission corridor: generation
at one end, load in the middle, an interconnector at the far end.

A line is described by its series **impedance** $z = R + jX$, or equivalently by
its **admittance** $y = 1/z$. X is about eight times R, which is typical of
transmission lines. The line parameters are invented. Say so whenever you quote
a result from this exercise set.

---

## 1 · The adjacency matrix

A graph is a set of nodes and a set of edges. The adjacency matrix is the
bookkeeping: $A_{ij} = 1$ if there is an edge from $i$ to $j$, and 0 otherwise.

### Your turn

Build it from the edge list. Do not call `core.six_bus_adjacency()` — write it,
then compare against it.

In [ ]:
# TODO 1 --- the adjacency matrix from the edge list ------------------------------------------
# Two `...` to replace, inside the loop (a line carries power both ways):
#   line 1  ->  A[a, b] = 1.0
#   line 2  ->  A[b, a] = 1.0
A = np.zeros((6, 6))
for (a, b) in edges:
    A[a, b] = 1.0
    A[b, a] = 1.0
# ------------------------------------------------------------------------------

In [ ]:
print("A =")
print(A.astype(int))
print("\nmatches core.six_bus_adjacency():", np.array_equal(A, core.six_bus_adjacency()))
print("symmetric      :", np.array_equal(A, A.T))
print("zero diagonal  :", np.all(np.diag(A) == 0))
print("edges counted  :", int(A.sum() // 2), "  (should be", len(edges), ")")
print("degrees        :", A.sum(axis=1).astype(int))

core.show_matrix(A, title="Adjacency matrix", fmt="{:.0f}", labels=core.BUS_NAMES)
plt.show()

**What you should see.** `matches ... True`, `symmetric : True`, `zero
diagonal : True`, `edges counted : 8`, and degrees `[2 3 3 3 3 2]`.

Three properties, three modelling decisions hiding inside them.

**Symmetric** because a transmission line is bidirectional. A graph of a process
plant with one-way flows would not be, and then `A[i, j]` and `A[j, i]` mean
different things and every formula below needs care about which index is which.

**Zero diagonal** because a bus is not its own neighbour. A layer that should
also use a bus's own state has to be given it separately, which is what the
layer in section 4 does.

**Unweighted** — every entry is 0 or 1, so the adjacency matrix says a line
exists but not how strong it is. The line impedances are left out. That is a
real modelling choice: the impedances belong in a separate table, the **edge
features** E, one row per line, which notebook 03 shows beside A. The graph
network in this exercise set uses A only, so that the arithmetic stays visible.

---

## 2 · Neighbourhoods and reach

The neighbourhood $\mathcal{N}(i)$ of node $i$ is the set of nodes joined to it
by an edge. It is the only thing a message-passing layer is allowed to look at.

### Your turn

Write `neighbours(A, i)` and use it to list the neighbourhood of every bus. Then
answer the question that decides how deep the network in notebook 03 has to be:
**how many hops does it take to get from any bus to any other?**

Powers of the adjacency matrix answer that. $(A^k)_{ij}$ counts the walks of
length exactly $k$ from $i$ to $j$; so the smallest $k$ for which
$(I + A + A^2 + \dots + A^k)_{ij} > 0$ is the distance between $i$ and $j$.

In [ ]:
# TODO 2 --- neighbours, and the diameter ---------------------------------------------------
# Two `...` to replace:
#   line 1  ->  np.nonzero(A[i])[0]                    the indices where row i is non-zero
#   line 2  ->  reach + np.linalg.matrix_power(A, k)   add the walks of length k
def neighbours(A, i):
    return np.nonzero(A[i])[0]

reach = np.eye(6)
for k in range(1, 7):
    reach = reach + np.linalg.matrix_power(A, k)
    if np.all(reach > 0):
        diameter = k
        break
# ------------------------------------------------------------------------------

In [ ]:
for i in range(core.N_BUS):
    nb = neighbours(A, i)
    print(f"  bus {i} ({core.BUS_NAMES[i]:11s}) neighbours: {nb.tolist()}")

print("\nA @ A (walks of length two):")
print((A @ A).astype(int))
print("\ndiameter:", diameter, "hops")

**What you should see.** Neighbour lists

```
bus 0 neighbours: [1, 2]
bus 1 neighbours: [0, 2, 3]
bus 2 neighbours: [0, 1, 4]
bus 3 neighbours: [1, 4, 5]
bus 4 neighbours: [2, 3, 5]
bus 5 neighbours: [3, 4]
```

the matrix $A^2$, and `diameter: 3 hops`.

The diagonal of $A^2$ is the degree of each node — a walk of length two that
returns to where it started must go out along an edge and back along the same
one. The off-diagonal entries count common neighbours: $(A^2)_{03} = 2$ says
buses 0 and 3 are joined by two different two-hop routes, through bus 1 and
through bus 2.

**The diameter is the number to remember.** It is three, so a node needs at least
three message-passing layers before it can be influenced by the farthest bus at
all. A one-layer graph network on this system is structurally incapable of
letting bus 0 know anything about bus 5, no matter how long you train it or how
wide you make it. Notebook 03 measures exactly this. In power-system terms: *with too few layers, a generator cannot feel a fault several buses away within one forward pass.*

Depth in a graph network is a statement about physics, not a hyperparameter you
tune blindly.

---

## 3 · Node features, and one round of message passing

Each node carries a feature vector. Stack them and you have $H$, of shape
(nodes, features). For now, give each bus a single feature so that the
arithmetic is checkable by hand: bus 0 gets a 1 and everybody else gets a 0.

The general message-passing rule from L5.1 is

$$\mathbf{h}_i^{(l+1)} = \phi\!\left(\mathbf{h}_i^{(l)},\;
\sum_{j\in\mathcal{N}(i)} \psi\!\left(\mathbf{h}_i^{(l)},\mathbf{h}_j^{(l)}\right)\right)$$

which reads: *build a message from each neighbour, add the messages up, and
combine the total with your own state.* Every graph network in the literature is
this equation with particular choices of $\psi$ and $\phi$.

The simplest possible choice — $\psi$ is "send your own state", $\phi$ is "take
the mean of what arrived" — is enough to see the mechanism.

### Your turn

Implement one round of mean aggregation over neighbours, with two loops, and then
show that the same thing is one matrix product.

In [ ]:
# TODO 3 --- one round of mean aggregation, twice --------------------------------------------
# Two `...` to replace:
#   line 1  ->  h[neighbours(A, i)].mean()          the mean of the neighbours' values
#   line 2  ->  D_inv @ A @ h                       the same, as one matrix product
h0 = np.zeros((6, 1))
h0[0, 0] = 1.0

def aggregate_loops(A, h):
    out = np.zeros_like(h)
    for i in range(A.shape[0]):
        out[i] = h[neighbours(A, i)].mean()
    return out

def aggregate_matrix(A, h):
    D_inv = np.diag(1.0 / A.sum(axis=1))
    return D_inv @ A @ h

h_loops  = aggregate_loops(A, h0)
h_matrix = aggregate_matrix(A, h0)
# ------------------------------------------------------------------------------

In [ ]:
print("h0        :", h0.ravel())
print("loops     :", h_loops.ravel())
print("matrices  :", h_matrix.ravel())
print("agree     :", np.allclose(h_loops, h_matrix))

h = h0.copy()
history = [h.ravel().copy()]
for step in range(4):
    h = aggregate_matrix(A, h)
    history.append(h.ravel().copy())

print("\nsignal starting at bus 0, spreading:")
print("  step      " + "".join(f"  bus {i}" for i in range(6)))
for s, row in enumerate(history):
    print(f"  {s:4d}      " + "".join(f"{v:7.3f}" for v in row))

fig, axes = plt.subplots(1, 4, figsize=(17.0, 3.4))
for s in range(4):
    core.plot_graph(A, node_values=history[s], ax=axes[s],
                    title=f"after {s} hops", cmap="viridis")
plt.show()

**What you should see.** `agree : True`, and a table in which the 1 that
started at bus 0 spreads outwards:

```
  step        bus 0  bus 1  bus 2  bus 3  bus 4  bus 5
     0        1.000  0.000  0.000  0.000  0.000  0.000
     1        0.000  0.333  0.333  0.000  0.000  0.000
     2        0.333  0.111  0.111  0.111  0.111  0.000
     3        0.111  0.185  0.185  0.074  0.074  0.111
     4        0.185  0.123  0.123  0.123  0.123  0.074
```

Four things are visible in that table, and each is a design problem in real graph
networks.

**Information travels one hop per layer.** After one round only buses 1 and 2 —
the neighbours of bus 0 — know anything. Bus 5, three hops away, is still exactly
zero after two rounds and first hears something at round three. That is the
diameter from section 2, appearing as a fact about the computation rather than
about the graph.

**Node 0 loses its own value immediately.** At step 1 bus 0 is zero, because the
rule as written averages the *neighbours* and forgets the node itself. That is
almost never what you want. The fix is to give the node's own state a term of
its own, next to the neighbours' sum — the layer in section 4 does exactly that.

**The total is not conserved.** Mean aggregation is not a physical transport
process, and nothing here is a conservation law. If you need one, you must put it
in the loss, which is what Part 2 spends twelve exercises doing.

**It flattens.** Keep going and every node converges towards the same value. This
is **over-smoothing**, and it is the reason deep graph networks are hard: after
enough rounds of averaging, every node has the graph's average and nothing of its
own. Depth in a graph network buys reach and costs distinctiveness, and the two
have to be traded against each other.

---

## 4 · Permutation: the property that makes it a graph network

Here is the property, from L5.1:

$$f(\mathbf{P}\mathbf{X},\; \mathbf{P}\mathbf{A}\mathbf{P}^\top)
= \mathbf{P}\, f(\mathbf{X}, \mathbf{A})$$

In words: **relabel the nodes, and the answer is relabelled the same way.**
Nothing about the computation depends on the numbers you happened to write on
the buses.

This is worth being exact about, because two words get used interchangeably and
they are not the same thing.

* **Equivariant**: the output moves with the input. A node-level graph network is
  equivariant — that is the equation above.
* **Invariant**: the output does not move at all. A *graph*-level network — one
  that pools all the nodes into a single prediction, "will this network be
  stable, yes or no" — is invariant, because the pooling step throws the ordering
  away.

The function $f$ here is one **message-passing layer**, the same layer notebook 03
trains:

$$\mathbf{H}' = \tanh\!\left(\mathbf{H}\,\mathbf{W}_{\mathrm{self}}
\;+\; \mathbf{A}\,\mathbf{H}\,\mathbf{W}_{\mathrm{neigh}}\right)$$

Row $i$ of $\mathbf{A}\mathbf{H}$ is the sum of bus $i$'s neighbours' features —
section 3's aggregation, summed rather than averaged. $\mathbf{W}_{\mathrm{self}}$
maps a bus's own features and $\mathbf{W}_{\mathrm{neigh}}$ the neighbours' sum;
both are shared by every bus. The weights here are random: no training.

### Your turn

Take the permutation `perm = [3, 0, 5, 2, 4, 1]`, meaning *the node now called 0
is the one that used to be called 3*, build its permutation matrix, and check
that permuting the inputs permutes the output of the layer.

In [ ]:
# TODO 4 --- permutation equivariance --------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  layer(A_perm, H_perm)      f(PX, P A P^T): the layer on the relabelled graph
#   line 2  ->  P @ layer(A, H)            P f(X, A): the old answer, relabelled
rng = np.random.default_rng(0)
W_self  = rng.normal(0.0, 0.5, size=(3, 4))       # 3 input features -> 4 output features
W_neigh = rng.normal(0.0, 0.5, size=(3, 4))
H = np.array([[1.0, 0.0, 0.0],                     # one feature row per bus, shape (6, 3)
              [0.0, 1.0, 0.0],
              [0.0, 0.0, 1.0],
              [1.0, 1.0, 0.0],
              [0.0, 1.0, 1.0],
              [1.0, 0.0, 1.0]])

def layer(A, H):
    # one message-passing layer: own features plus the sum over neighbours
    return np.tanh(H @ W_self + A @ H @ W_neigh)

perm = [3, 0, 5, 2, 4, 1]
P    = core.permutation_matrix(perm)              # P[i, perm[i]] = 1

A_perm = P @ A @ P.T
H_perm = P @ H

left  = layer(A_perm, H_perm)
right = P @ layer(A, H)
gap = float(np.abs(left - right).max())
# ------------------------------------------------------------------------------

In [ ]:
print("H shape", H.shape, "-> layer output shape", layer(A, H).shape)
print("\npermutation:", perm)
print("\nP =")
print(P.astype(int))
print("\ndegrees before:", A.sum(axis=1).astype(int))
print("degrees after :", A_perm.sum(axis=1).astype(int))
print("\nlargest |f(PX, PAP^T) - P f(X, A)| =", gap)
print("equivariant to machine precision:", gap < 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.0))
core.plot_graph(A, ax=axes[0], title="original labelling",
                labels=[str(i) for i in range(6)])
core.plot_graph(A_perm, ax=axes[1], title="after relabelling",
                labels=[str(i) for i in range(6)])
plt.show()

**What you should see.** `H shape (6, 3) -> layer output shape (6, 4)`, the
permutation matrix, degrees `[2 3 3 3 3 2]` becoming `[3 2 2 3 3 3]`, and

```
largest |f(PX, PAP^T) - P f(X, A)| = 0.0
equivariant to machine precision: True
```

Exactly zero. This is not an approximation that
training improved; it is an algebraic identity that holds for **any** weights,
trained or random, because the only operations involved are a per-node linear map
and a sum over neighbours, and neither can see a node index.

**The shape went from (6, 3) to (6, 4).** Six nodes in, six nodes out. A
message-passing layer never changes the number of nodes; it changes how many
numbers each node carries. That is what makes it a *node-level* operation, and
it is why the task in notebook 03 can be node regression.

Compare that with what a dense network on the flattened state vector would do.
Feed it the six buses in a different order and you have fed it a different input,
so you get a different answer. It could learn to be permutation invariant, from
data, approximately, for the orderings it happened to see — which is why graph networks earn their place on a power system.
Notebook 03 measures the difference rather than asserting it.

The two pictures show why this matters practically. They are the same network.
Only the numbers written on the buses changed, and no engineering conclusion may
depend on those numbers.

---

## 5 · The parameter count, one more time

The last thing to notice before training anything: **the size of the weights
does not depend on the number of nodes.**

`W_self` and `W_neigh` above are 3 by 4 each — three input features, four output
features — and they would be 3 by 4 on a six-bus network, on the Danish
transmission system, or on a single isolated bus. That is weight sharing across
nodes, and it is the same argument as the 80 parameters in notebook 01.

In [ ]:
for n_nodes in (6, 60, 600):
    dense = (n_nodes * 3) * (n_nodes * 4) + n_nodes * 4    # weights + biases
    mp = 3 * 4 + 3 * 4                                      # W_self and W_neigh
    print(f"  {n_nodes:4d} nodes:  dense layer {dense:9,d}   "
          f"message passing {mp:4d}   ratio {dense / mp:9,.0f} x")

**What you should see.**

```
     6 nodes:  dense layer       456   message passing   24   ratio        19 x
    60 nodes:  dense layer    43,440   message passing   24   ratio     1,810 x
   600 nodes:  dense layer 4,322,400   message passing   24   ratio   180,100 x
```

The message-passing column is constant. That is what lets a model trained on
small cases say anything at all about a larger system — the "one model, many
grids" argument, and the reason the architecture is worth the
trouble even when the topology is already known exactly.

---

## 6 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The adjacency matrix of the six-bus network records that a line exists but not its impedance. Name a prediction on this network where leaving the impedances out would matter, and say how the line data could be given to a graph network alongside the adjacency.
   *→ L5.1 Q6*
2. In section 3 a value placed at bus 0 spread one hop per round and then flattened towards a common value. Say what one round of message passing does, what over-smoothing is, and how the graph's diameter of three sets the smallest number of layers a graph network on this network needs.
   *→ L5.1 Q7, Q8*
3. The permutation check gave a gap of exactly zero with random, untrained weights. Say in your own words what permutation equivariance guarantees, and why its holding for random weights shows that it comes from the layer rather than from training.
   *→ L5.1 Q9*
4. The message-passing layer has 24 weights however many buses there are, while a dense layer on the same buses has 456 at six buses and over four million at six hundred. Explain where the difference comes from, and relate it to weight sharing in a convolution on an image.
   *→ L5.1 Q9*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.
